This File exists to be the main file for my work

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import getpass

from dotenv import load_dotenv
from configparser import ConfigParser
from sqlalchemy import create_engine

import os

In [17]:
# We seek to use a config file to avoid absolute file paths
file = 'config.ini' #config file of interest

config = ConfigParser()
config.read(file)

print(config.sections()) # outlines the sections in the file

['paths', 'postgres']


In [18]:
# accesses the motor data element, in this case it's the file path for it
paths = [config['paths']['motor_data'], config['paths']['claims_data']]

#path
df_motor = pd.read_csv(paths[0], sep=";")
df_claims = pd.read_csv(paths[1], sep=";")

In [19]:
df_motor.columns.tolist(), df_claims.columns.tolist() # It's important to understand the columns that exist in the tables

(['ID',
  'Date_start_contract',
  'Date_last_renewal',
  'Date_next_renewal',
  'Date_birth',
  'Date_driving_licence',
  'Distribution_channel',
  'Seniority',
  'Policies_in_force',
  'Max_policies',
  'Max_products',
  'Lapse',
  'Date_lapse',
  'Payment',
  'Premium',
  'Cost_claims_year',
  'N_claims_year',
  'N_claims_history',
  'R_Claims_history',
  'Type_risk',
  'Area',
  'Second_driver',
  'Year_matriculation',
  'Power',
  'Cylinder_capacity',
  'Value_vehicle',
  'N_doors',
  'Type_fuel',
  'Length',
  'Weight'],
 ['ID', 'Cost_claims_year', 'Cost_claims_by_type', 'Claims_type'])

In [20]:
df_motor

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Area,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight
0,1,05/11/2015,05/11/2015,05/11/2016,15/04/1956,20/03/1976,0,4,1,2,...,0,0,2004,80,599,7068.00,0,P,NaN,190
1,1,05/11/2015,05/11/2016,05/11/2017,15/04/1956,20/03/1976,0,4,1,2,...,0,0,2004,80,599,7068.00,0,P,NaN,190
2,1,05/11/2015,05/11/2017,05/11/2018,15/04/1956,20/03/1976,0,4,2,2,...,0,0,2004,80,599,7068.00,0,P,NaN,190
3,1,05/11/2015,05/11/2018,05/11/2019,15/04/1956,20/03/1976,0,4,2,2,...,0,0,2004,80,599,7068.00,0,P,NaN,190
4,2,26/09/2017,26/09/2017,26/09/2018,15/04/1956,20/03/1976,0,4,2,2,...,0,0,2004,80,599,7068.00,0,P,NaN,190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53498,30/07/2018,30/07/2018,30/07/2019,25/07/1981,14/02/2007,0,1,1,1,...,0,0,2000,110,1997,24320.00,5,D,4.740,1480
105551,53499,16/08/2018,16/08/2018,16/08/2019,08/12/1976,29/11/2017,0,1,1,1,...,0,0,2013,129,1998,30861.97,5,P,4.650,1440
105552,53500,21/11/2018,21/11/2018,21/11/2019,01/04/1974,05/10/2011,0,1,1,1,...,0,0,1999,55,999,7800.00,5,P,3.495,830
105553,53501,21/11/2018,21/11/2018,21/11/2019,15/09/1946,02/02/1982,0,1,1,1,...,0,0,2004,90,1753,16610.00,5,D,4.555,1399


In [48]:
df_motor['Date_last_renewal'] = pd.to_datetime(df_motor['Date_last_renewal'], format='%d/%m%Y')

In [49]:
df_motor['Date_last_renewal']

0        2015-11-05
1        2016-11-05
2        2017-11-05
3        2018-11-05
4        2017-09-26
            ...    
105550   2018-07-30
105551   2018-08-16
105552   2018-11-21
105553   2018-11-21
105554   2018-10-01
Name: Date_last_renewal, Length: 105555, dtype: datetime64[ns]

In [21]:
df_claims.head()

,ID,Cost_claims_year,Cost_claims_by_type,Claims_type
0,28,332.02,332.02,complaint
1,36,57.35,57.35,travel assistance
2,42,57.50,57.50,travel assistance
3,60,138.90,67.25,travel assistance
4,60,138.90,71.65,travel assistance


#### Now that we have the data set of interest, it's useful for us to outline the questions of interest.

- Who claims?
    - What is there demographic?

- How much do they cost?

- What variables drive risk?

- What should the insurer charge?

To answer these questions it would be important that we correctly format and clean the data.

In [22]:
#df_motor.dropna(subset=['Length','Date_lapse'])
df_motor.isna().sum() # checks the number of  Na values in the columns

ID                          0
Date_start_contract         0
Date_last_renewal           0
Date_next_renewal           0
Date_birth                  0
Date_driving_licence        0
Distribution_channel        0
Seniority                   0
Policies_in_force           0
Max_policies                0
Max_products                0
Lapse                       0
Date_lapse              70408
Payment                     0
Premium                     0
Cost_claims_year            0
N_claims_year               0
N_claims_history            0
R_Claims_history            0
Type_risk                   0
Area                        0
Second_driver               0
Year_matriculation          0
Power                       0
Cylinder_capacity           0
Value_vehicle               0
N_doors                     0
Type_fuel                1764
Length                  10329
Weight                      0
dtype: int64

In [23]:
# checks if the three columns found previously are actually the only columns on interest in this regard.
df_motor.isnull().any()
df_motor_nulls = df_motor.loc[:, df_motor.isnull().any()]
print(df_motor_nulls)

       Date_lapse Type_fuel  Length
0             NaN         P     NaN
1             NaN         P     NaN
2             NaN         P     NaN
3             NaN         P     NaN
4             NaN         P     NaN
...           ...       ...     ...
105550        NaN         D   4.740
105551        NaN         P   4.650
105552        NaN         P   3.495
105553        NaN         D   4.555
105554        NaN         D   4.854

[105555 rows x 3 columns]


In [24]:
df_motor.info(memory_usage = 'Deep') # can use True instead if the calculation is too computationally intensitive

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105555 entries, 0 to 105554
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   ID                    105555 non-null  int64  
 1   Date_start_contract   105555 non-null  object 
 2   Date_last_renewal     105555 non-null  object 
 3   Date_next_renewal     105555 non-null  object 
 4   Date_birth            105555 non-null  object 
 5   Date_driving_licence  105555 non-null  object 
 6   Distribution_channel  105555 non-null  int64  
 7   Seniority             105555 non-null  int64  
 8   Policies_in_force     105555 non-null  int64  
 9   Max_policies          105555 non-null  int64  
 10  Max_products          105555 non-null  int64  
 11  Lapse                 105555 non-null  int64  
 12  Date_lapse            35147 non-null   object 
 13  Payment               105555 non-null  int64  
 14  Premium               105555 non-null  float64
 15  

In [25]:
df_claims.info() # from this we can say that the claims dataframe doesn't need missing values handling.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7366 entries, 0 to 7365
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   7366 non-null   int64  
 1   Cost_claims_year     7366 non-null   float64
 2   Cost_claims_by_type  7366 non-null   float64
 3   Claims_type          7366 non-null   object 
dtypes: float64(2), int64(1), object(1)
memory usage: 230.3+ KB


Now a key detail to understand is that there's two main ways to handle missing values, either removal of the data point of issue or the substitution of those said data points. (1)[Kwak, S. G., & Kim, J. H. (2017). Statistical data preparation: management of missing values and outliers. Korean Journal of Anesthesiology, 70(4), 407–411. https://doi.org/10.4097/kjae.2017.70.4.407.]

To know which to perform it is important to quantify the statistical significance of removing the offending data points. A good way of checking this is finding if there's a pattern within the missingness or if the missingness is simply random. 

To properly check this we will be using Little’s MCAR test. After which specific trends of interest could be extracted using further statistical methods.

#### Algorithm Behind Little’s MCAR Test

- Group Data by Missingness Patterns: The dataset is divided into groups based on missing data patterns (e.g., one group where variable X is missing, another where Y is missing, etc.).

- Calculate Means and Covariances for Each Group: Compute the mean vectors and covariance matrices for each missing data group.

- Estimate Expected Mean Differences: Using maximum likelihood estimation (MLE), the test estimates the expected mean for each variable based on available data.

- Compute the Test Statistic: The test calculates a Chi-Square statistic by comparing the estimated means across missing data patterns.

- Compare with the Chi-Square Distribution: The calculated Chi-Square value is compared with a theoretical Chi-Square distribution to determine significance (p-value).

#### Decision Rule:

- p > 0.05 → Fail to reject the null hypothesis → Data is MCAR (missing completely at random).

- p ≤ 0.05 → Reject the null hypothesis → Data is MAR or MNAR, meaning missingness follows a pattern.

In [26]:
#df_length_fuel_lapse = df_motor[['ID', 'Length', 'Type_fuel', 'Date_lapse']].isna()
#df_length_fuel_lapse.value_counts()

missing = df_motor[['Length', 'Type_fuel', 'Date_lapse']].isna().any(axis=1)

result = df_motor.loc[missing, ['ID', 'Length', 'Type_fuel', 'Date_lapse']]

unique_result = result.drop_duplicates('ID')

len(unique_result)/len(df_motor) * 100

33.31533323859599

With 33.32% missingness, it is very unlikely that the data can simply be drop as the impact will be significant.

Due to this we will use machine learning classification to perform this fix.

In [28]:
#df_motor['Date_lapse'] = pd.to_datetime(df_motor['Date_lapse']) , # doesn't work rn due to Nan values

In [29]:
train = df_motor[df_motor[['Length', 'Type_fuel', 'Date_lapse']].notna()]
train[['Length', 'Type_fuel', 'Date_lapse']]

,Length,Type_fuel,Date_lapse
0,NaN,P,NaN
1,NaN,P,NaN
2,NaN,P,NaN
3,NaN,P,NaN
4,NaN,P,NaN
...,...,...,...
105550,4.740,D,NaN
105551,4.650,P,NaN
105552,3.495,P,NaN
105553,4.555,D,NaN


In [50]:
df_motor['Type_fuel'].unique()

array(['P', 'D', nan], dtype=object)

In [27]:
df_motor.columns # we inspect the columns here to find the potential features of interest

Index(['ID', 'Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
       'Date_birth', 'Date_driving_licence', 'Distribution_channel',
       'Seniority', 'Policies_in_force', 'Max_policies', 'Max_products',
       'Lapse', 'Date_lapse', 'Payment', 'Premium', 'Cost_claims_year',
       'N_claims_year', 'N_claims_history', 'R_Claims_history', 'Type_risk',
       'Area', 'Second_driver', 'Year_matriculation', 'Power',
       'Cylinder_capacity', 'Value_vehicle', 'N_doors', 'Type_fuel', 'Length',
       'Weight'],
      dtype='object')

In [40]:
df_motor[['Date_birth']].mean()

TypeError: Could not convert ['...'] to numeric

In [ ]:
# I will drop the date eliments as for this process, it will likely
df_motor.drop(
    columns=['ID','Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
       'Date_birth', 'Date_driving_licence', 'Date_lapse', 'Cost_claims_year'])

In [ ]:
# The first step here we should likely choose to select Length as the missing variable of interest, then work to see if we can predict that missingness based upon the others then iterate.
Mo = df_motor[['ID','Seniority','Policies_in_force']].mean()

np.float64(4.252007487450906)

### Here we will make a database using postgreSQL

We've downloaded the stuff and installed it idk what's happening now after that tho, we should figure that out and make the database work

In [31]:
# connection to db

#load_dotenv() # Loads the env file which holds the password to the postgres database

#pgcon = psycopg2.connect(host = config['postgres']['host'],
#                         user = f"{config['postgres']['username']}", # accesses the username of the current user
#                         password = os.getenv("POSTGRES_PASSWORD"), # grabs the password from the env file
#                         database = config['postgres']['database_type'])